In [1]:
import duckdb
import numpy as np
import pandas as pd

df = duckdb.sql("SELECT * FROM 'data/features_50k.parquet'").df()
df = df.sort_values(["date_time", "srch_id"]).reset_index(drop=True)
df.shape

(1239782, 69)

In [2]:
search_time = df.groupby("srch_id")["date_time"].first()
t1, t2 = search_time.quantile([0.7, 0.8])

df["split"] = np.where(df["date_time"] < t1, "train",
              np.where(df["date_time"] < t2, "valid", "test"))

df.groupby("split").agg(
    searches=("srch_id", "nunique"),
    rows=("srch_id", "size"),
    start=("date_time", "min"),
    end=("date_time", "max"),
)

,searches,rows,start,end
split,,,,
test,10000,243583,2013-05-21 12:13:02,2013-06-30 23:38:23
train,35000,873624,2012-11-01 01:41:51,2013-04-29 17:50:34
valid,5000,122575,2013-04-29 17:53:17,2013-05-21 12:11:14


In [3]:
def ndcg_at_k(data, score_col, k=5):
    def one_search(g):
        ranked = g.sort_values(score_col, ascending=False)
        gains = 2 ** ranked["relevance"].to_numpy()[:k] - 1
        discounts = np.log2(np.arange(2, len(gains) + 2))
        dcg = (gains / discounts).sum()

        ideal = np.sort(2 ** g["relevance"].to_numpy() - 1)[::-1][:k]
        idcg = (ideal / discounts).sum()
        return dcg / idcg if idcg > 0 else np.nan

    return data.groupby("srch_id").apply(one_search).mean()

In [4]:
test = df[df["split"] == "test"].copy()

rng = np.random.default_rng(42)
test["score_random"]  = rng.random(len(test))
test["score_price"]   = -test["price_usd"]
test["score_review"]  = test["prop_review_score"].fillna(0)
test["score_loc2"]    = test["prop_location_score2"].fillna(0)
test["score_expedia"] = -test["position"]

baselines = {
    "random": "score_random",
    "cheapest first": "score_price",
    "best reviewed first": "score_review",
    "best location first": "score_loc2",
    "Expedia's ranking (reference)": "score_expedia",
}

results = {name: ndcg_at_k(test, col) for name, col in baselines.items()}
pd.Series(results).sort_values().round(4)

random                           0.1619
best reviewed first              0.1695
cheapest first                   0.2091
best location first              0.2782
Expedia's ranking (reference)    0.3775
dtype: float64